# Differentiable Monte Carlo — The Real Deal

**Goal:** Convert the paper's MC simulation into a fully differentiable pipeline in PyTorch.

**Plan (step by step):**
1. Implement the full MC simulation as a PyTorch module
2. Make the sampling differentiable (from toy-experiment-sampling)
3. Make the Voigt fit differentiable via implicit differentiation (from toy-experiment-fitting)
4. Compare simulated vs experimental distributions using smooth density comparison
5. Optimize (γ, n̄) via gradient descent

---

## Step 1: Imports & Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass

print('Ready')

## Step 2: One MC Run 

One run = simulate one PLE scan with **continuous photon positions** .

Steps:
1. Sample total photon count $n \sim \mathcal{N}(\bar{n}, \sigma)$
2. Draw $n$ photon frequencies from the Cauchy($\gamma$) lineshape (continuous)
3. Add background noise: draw $n_{\text{bg}} \sim \text{Poisson}(\lambda)$, distribute uniformly
4. **Return:** all photon frequencies concatenated into one tensor

Later: fit a Voigt to these continuous samples to extract one linewidth $w_i$.

In [ ]:
@dataclass
class MCParams:
    gamma: float       # HWHM of Cauchy (MHz) — optimized
    nbar: float        # mean photon count — optimized
    sigma: float = 6.0  # noise std (fixed, from paper)
    lambda_: float = 2.0  # mean background counts (fixed, from paper)

# Frequency window (from paper)
FREQ_MIN = -75.0   # MHz
FREQ_MAX = 75.0    # MHz
print(f'Frequency window: [{FREQ_MIN}, {FREQ_MAX}] MHz')

In [ ]:
def sample_photon_count(nbar, sigma, epsilon):
    """Reparameterized sampling of total photon count (non-negative int)."""
    n_float = nbar + sigma * epsilon
    return max(round(n_float), 0)

def one_run(params, epsilon, rng=None):
    """
    One MC run simulating one PLE scan.
    
    Returns:
        signal_freqs: tensor of shape (n,) — signal photon frequencies
        bg_freqs: tensor of shape (n_bg,) — background photon frequencies
        (Concatenated they form the full "measured" spectrum as point cloud)
    """
    if rng is None:
        rng = np.random.default_rng()
    
    # 1. Sample total signal photon count
    n = sample_photon_count(params.nbar, params.sigma, epsilon)
    
    # 2. Sample n continuous photon frequencies from Cauchy(gamma)
    #    Cauchy can be sampled via: gamma * tan(pi*(u - 0.5)) where u ~ Uniform(0,1)
    if n > 0:
        u = rng.uniform(0, 1, n)
        samples = params.gamma * np.tan(np.pi * (u - 0.5))
        # Clip to frequency window
        signal_freqs = np.clip(samples, FREQ_MIN, FREQ_MAX)
    else:
        signal_freqs = np.array([])
    
    # 3. Background noise: uniform across window
    n_bg = rng.poisson(params.lambda_)
    if n_bg > 0:
        bg_freqs = rng.uniform(FREQ_MIN, FREQ_MAX, n_bg)
    else:
        bg_freqs = np.array([])
    
    return (
        torch.tensor(signal_freqs, dtype=torch.float32),
        torch.tensor(bg_freqs, dtype=torch.float32),
    )


# Quick test
params = MCParams(gamma=15.0, nbar=40.0)
eps = np.random.normal()
signal, bg = one_run(params, eps)

print(f'Sample eps = {eps:.2f}')
print(f'Photon count n = {len(signal)}  (nbar={params.nbar})')
print(f'Background events n_bg = {len(bg)}  (lambda={params.lambda_})')
print(f'Signal freq range: [{signal.min().item():.1f}, {signal.max().item():.1f}] MHz')
print(f'Signal freqs (first 10): {signal[:10].tolist()}')

The key change: no bins, no histogram. Each run gives us a **point cloud** of detected photon frequencies.

Next up: **fitting a Voigt** to each run's point cloud to extract the FWHM $w_i$.